In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 1 — Idle clip (boomerang with EASED turnaround, NO audio)
# Plays a forward slice, then its reverse, so the clip starts and ends on the
# same frame (seamless loop). To smooth the reversal, the last bit of the
# forward slice is SLOWED DOWN, so motion decelerates into the turnaround:
#     forward  =  [0 : NORMAL]s at normal speed
#               +  next EASE_SRC s  stretched to play over EASE_OUT s  (slow)
#     idle     =  forward  +  reverse(forward)
#     total    =  2 * (NORMAL + EASE_OUT)
#   e.g. NORMAL=1.4, EASE_SRC=0.2, EASE_OUT=0.3  ->  3.4s, smooth turnaround
#   (Set EASE_SRC=0 for a plain hard turnaround.)
# ════════════════════════════════════════════════════════════
# from google.colab import drive; drive.mount('/content/drive')   # uncomment on Colab

import os, json, subprocess
from pathlib import Path

# ── EDIT THESE ───────────────────────────────────────────────
IDLE_ANIMATION = './assets/person_03_idle.mp4'  # your subtle idle render
OUTPUT_PATH    = './assets/person_03_idle_3s.mp4'
NORMAL_SEC     = 1.4    # forward portion played at normal speed
EASE_SRC_SEC   = 0.2    # source seconds just before the turnaround to slow down
EASE_OUT_SEC   = 0.3    # play those EASE_SRC seconds over this many seconds (>EASE_SRC = slower)
# ─────────────────────────────────────────────────────────────

def _duration(path):
    out = subprocess.run(['ffprobe', '-v', 'error', '-show_entries', 'format=duration',
                          '-of', 'json', str(path)], capture_output=True, text=True).stdout
    return float(json.loads(out)['format']['duration'])

def _fps(path):
    out = subprocess.run(['ffprobe', '-v', 'error', '-select_streams', 'v:0',
                          '-show_entries', 'stream=r_frame_rate', '-of', 'json',
                          str(path)], capture_output=True, text=True).stdout
    n, d = json.loads(out)['streams'][0]['r_frame_rate'].split('/')
    return float(n) / float(d)

def make_boomerang(src, out, normal_sec, ease_src_sec=0.0, ease_out_sec=0.0):
    """forward (with optional slowed tail) + reverse(forward), video only (no audio)."""
    src, out = str(src), str(out)
    if not os.path.exists(src):
        raise FileNotFoundError(f'Input not found: {src}')
    os.makedirs(os.path.dirname(out) or '.', exist_ok=True)
    fps = _fps(src)
    eased = ease_src_sec > 0 and ease_out_sec > 0
    need  = normal_sec + (ease_src_sec if eased else 0.0)
    src_dur = _duration(src)
    if src_dur < need:
        print(f'  WARNING: source is {src_dur:.2f}s (< {need:.2f}s needed) — clamping.')
        normal_sec = max(0.0, src_dur - (ease_src_sec if eased else 0.0))

    if eased:
        factor = ease_out_sec / ease_src_sec          # >1 = slower
        fwd = (f'[0:v]trim=0:{normal_sec},setpts=PTS-STARTPTS[h];'
               f'[0:v]trim={normal_sec}:{normal_sec + ease_src_sec},'
               f'setpts={factor:.6f}*(PTS-STARTPTS)[t];'
               f'[h][t]concat=n=2:v=1,setpts=PTS-STARTPTS[fwd]')
        half_out = normal_sec + ease_out_sec
    else:
        fwd = f'[0:v]trim=0:{normal_sec},setpts=PTS-STARTPTS[fwd]'
        half_out = normal_sec

    fc = f'{fwd};[fwd]split[a][b];[b]reverse[r];[a][r]concat=n=2:v=1,fps={fps:.6f}[v]'
    cmd = ['ffmpeg', '-y', '-i', src, '-filter_complex', fc,
           '-map', '[v]', '-an', '-c:v', 'libx264', '-pix_fmt', 'yuv420p', out]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError('ffmpeg failed:\n' + r.stderr[-1500:])
    print(f'  done -> {out}\n         {_duration(out):.2f}s (target {2*half_out:.2f}s), '
          f'{fps:.2f} fps, no audio')
    return out

print(f'Idle boomerang (eased turnaround)  ->  '
      f'2*({NORMAL_SEC}+{EASE_OUT_SEC}) = {2*(NORMAL_SEC+EASE_OUT_SEC):.2f}s '
      f'from {Path(IDLE_ANIMATION).name}')
make_boomerang(IDLE_ANIMATION, OUTPUT_PATH, NORMAL_SEC, EASE_SRC_SEC, EASE_OUT_SEC)


Idle boomerang (eased turnaround)  ->  2*(1.4+0.3) = 3.40s from person_08_idle.mp4
  done -> ./assets/person_08_idle_3s.mp4
         3.40s (target 3.40s), 30.00 fps, no audio


'./assets/person_08_idle_3s.mp4'

In [1]:
# ════════════════════════════════════════════════════════════
# CELL 2 — Speaking clip (EXACTLY S seconds, NO audio)
# Built from a SEAMLESS eased half-video boomerang, tiled to fill S:
#   unit  = first V/2 of the video, last EASE_SRC s slowed to EASE_OUT s,
#           then + its reverse, with the duplicate turnaround frame dropped.
#   fill  = N = floor(S / L) whole units (L = unit length), dropping the 1
#           duplicate frame at every join, then an eased boomerang of the
#           leftover R/2, then STRETCH the whole thing to exactly S.
#   • S <= ~V -> N=0 -> a single eased boomerang of the first S/2 (no loop).
#   • S  >  V -> seamless units repeat, remainder boomerang fills the rest.
# Starts and ends on frame 0 (neutral) so it joins cleanly to the idle clip.
# (Self-contained. Needs ffmpeg + ffprobe on PATH.)
# ════════════════════════════════════════════════════════════
import os, json, subprocess, tempfile
from pathlib import Path

# ── EDIT THESE ───────────────────────────────────────────────
TALKING_ANIMATION = './assets/person_03_talking.mp4'  # your talking render
OUTPUT_PATH       = './assets/person_03_speaking_Ss.mp4'
S_SECONDS         = 22.44   # exact total speaking length (loops the video as needed)
# Turnaround easing (slow into each reversal) — same idea as the idle cell:
EASE_SRC_SEC      = 0.2   # source seconds just before a turnaround...
EASE_OUT_SEC      = 0.3   # ...stretched to play over this long (>EASE_SRC = slower)
# ─────────────────────────────────────────────────────────────

def _dur(path):
    out = subprocess.run(['ffprobe', '-v', 'error', '-show_entries', 'format=duration',
                          '-of', 'json', str(path)], capture_output=True, text=True).stdout
    return float(json.loads(out)['format']['duration'])

def _fps(path):
    out = subprocess.run(['ffprobe', '-v', 'error', '-select_streams', 'v:0',
                          '-show_entries', 'stream=r_frame_rate', '-of', 'json',
                          str(path)], capture_output=True, text=True).stdout
    n, d = json.loads(out)['streams'][0]['r_frame_rate'].split('/')
    return float(n) / float(d)

def _ff(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError('ffmpeg failed:\n' + r.stderr[-1500:])

def _eased_forward(src, out, half, fps, e_src, e_out):
    """First `half` seconds of src, with the last e_src s slowed to e_out s."""
    half = float(half)
    normal = half - e_src
    if normal > 0 and e_src > 0 and e_out > 0:
        factor = e_out / e_src                      # >1 = slower
        fc = (f'[0:v]trim=0:{normal:.6f},setpts=PTS-STARTPTS[a];'
              f'[0:v]trim={normal:.6f}:{half:.6f},setpts={factor:.6f}*(PTS-STARTPTS)[b];'
              f'[a][b]concat=n=2:v=1,fps={fps:.6f}[v]')
    else:                                            # too short to ease — plain clip
        fc = f'[0:v]trim=0:{half:.6f},setpts=PTS-STARTPTS,fps={fps:.6f}[v]'
    _ff(['ffmpeg', '-y', '-i', str(src), '-filter_complex', fc, '-map', '[v]', '-an',
         '-c:v', 'libx264', '-pix_fmt', 'yuv420p', '-crf', '18', str(out)])
    return out

def _boomerang(fwd, out, fps, drop_first=False):
    """forward + reverse(forward), minus the duplicate turnaround frame.
    drop_first also removes the very first frame (for a seamless join after a
    previous piece that already ends on this same frame)."""
    base = ('[0:v]split[x][y];'
            '[y]reverse,trim=start_frame=1,setpts=PTS-STARTPTS[r];'
            f'[x][r]concat=n=2:v=1,fps={fps:.6f}')
    fc = (base + f',trim=start_frame=1,setpts=PTS-STARTPTS,fps={fps:.6f}[v]') if drop_first \
         else (base + '[v]')
    _ff(['ffmpeg', '-y', '-i', str(fwd), '-filter_complex', fc, '-map', '[v]', '-an',
         '-c:v', 'libx264', '-pix_fmt', 'yuv420p', '-crf', '18', str(out)])
    return out

def build_speaking(src, out, S, e_src, e_out):
    src, out = str(src), str(out)
    if not os.path.exists(src):
        raise FileNotFoundError(f'Input not found: {src}')
    os.makedirs(os.path.dirname(out) or '.', exist_ok=True)
    V, fps = _dur(src), _fps(src)
    tmp = tempfile.mkdtemp()
    posix = lambda p: str(p).replace('\\', '/')

    # Estimated unit (half-video eased boomerang) length, for the loop count.
    half_unit = V / 2.0
    L = (V + 2.0 * (e_out - e_src) - 1.0 / fps) if half_unit > e_src else (V - 1.0 / fps)
    N = max(0, int(S // L)) if L > 0 else 0
    R = S - N * L

    pieces = []
    if N >= 1:
        fwd_u = _eased_forward(src, f'{tmp}/fwd_u.mp4', half_unit, fps, e_src, e_out)
        unit_full  = _boomerang(fwd_u, f'{tmp}/unit_full.mp4',  fps, drop_first=False)
        pieces.append(unit_full)
        if N >= 2:
            unit_nodup = _boomerang(fwd_u, f'{tmp}/unit_nodup.mp4', fps, drop_first=True)
            pieces += [unit_nodup] * (N - 1)
        if R > 0.3:                                  # leftover worth its own boomerang
            fwd_r = _eased_forward(src, f'{tmp}/fwd_r.mp4', R / 2.0, fps, e_src, e_out)
            pieces.append(_boomerang(fwd_r, f'{tmp}/rem.mp4', fps, drop_first=True))
    else:                                            # S <= unit: one eased boomerang
        R = S
        fwd_r = _eased_forward(src, f'{tmp}/fwd_r.mp4', S / 2.0, fps, e_src, e_out)
        pieces.append(_boomerang(fwd_r, f'{tmp}/rem.mp4', fps, drop_first=False))

    # Concat all pieces (identical codec/params -> demuxer is safe).
    listf = f'{tmp}/list.txt'
    with open(listf, 'w') as f:
        for p in pieces:
            f.write(f"file '{posix(p)}'\n")
    concat_tmp = f'{tmp}/concat.mp4'
    _ff(['ffmpeg', '-y', '-f', 'concat', '-safe', '0', '-i', listf, '-an',
         '-c:v', 'libx264', '-pix_fmt', 'yuv420p', '-crf', '18', '-r', f'{fps:.6f}', concat_tmp])

    # Stretch the assembled clip to land on exactly S (absorbs dropped frames).
    D = _dur(concat_tmp)
    factor = S / D
    fc = f'[0:v]setpts={factor:.6f}*PTS,fps={fps:.6f}[v]'
    _ff(['ffmpeg', '-y', '-i', concat_tmp, '-filter_complex', fc, '-map', '[v]', '-an',
         '-c:v', 'libx264', '-pix_fmt', 'yuv420p', '-crf', '18', out])

    final = _dur(out)
    print(f'  V={V:.2f}s | unit~{L:.2f}s | S={S:.2f}s  ->  {N} seamless unit(s) '
          f'+ {R:.2f}s remainder boomerang')
    print(f'  pre-stretch {D:.2f}s -> stretched x{factor:.4f} -> {final:.2f}s, {fps:.2f} fps, no audio')
    print(f'  done -> {out}')
    return out

print(f'Speaking clip  ->  {S_SECONDS}s from {Path(TALKING_ANIMATION).name}')
build_speaking(TALKING_ANIMATION, OUTPUT_PATH, S_SECONDS, EASE_SRC_SEC, EASE_OUT_SEC)


Speaking clip  ->  22.44s from person_03_talking.mp4
  V=8.20s | unit~8.37s | S=22.44s  ->  2 seamless unit(s) + 5.71s remainder boomerang
  pre-stretch 22.57s -> stretched x0.9944 -> 22.43s, 30.00 fps, no audio
  done -> ./assets/person_03_speaking_Ss.mp4


'./assets/person_03_speaking_Ss.mp4'

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 3 — Stitch the full timeline (NO audio)
#     0.0s        ->  3.0s        : idle / static face   (Cell 1 output)
#     3.0s        -> (3.0 + S)s   : speaking animation   (Cell 2 output, length S)
#    (3.0 + S)s   -> (6.0 + S)s   : idle / static face   (same idle clip)
#   total = 6.0 + S
# The idle clip is stretched to EXACTLY IDLE_DURATION; the speaking clip is
# used at its own length (that IS S). Both are normalized to the speaking
# clip's fps + resolution so the joins are clean.
#   CROSSFADE_SEC = 0  -> hard cuts, exact 6+S.
#   CROSSFADE_SEC > 0  -> soft blends at the 2 joins (total = 6+S - 2*CF).
# (Self-contained. Needs ffmpeg + ffprobe on PATH. Run Cells 1 & 2 first.)
# ════════════════════════════════════════════════════════════
import os, json, subprocess, tempfile
from pathlib import Path

# ── EDIT THESE ───────────────────────────────────────────────
IDLE_CLIP     = './assets/person_03_idle_3s.mp4'       # Cell 1 output (idle boomerang)
SPEAKING_CLIP = './assets/person_03_speaking_Ss.mp4'   # Cell 2 output (S-second speaking)
OUTPUT_PATH   = './assets/person_03_sequenced.mp4'
IDLE_DURATION = 3.0     # seconds of idle at the start AND the end
CROSSFADE_SEC = 0.0     # 0 = hard cut (exact 6+S); >0 = blend the 2 joins
# ─────────────────────────────────────────────────────────────

def _dur(path):
    out = subprocess.run(['ffprobe', '-v', 'error', '-show_entries', 'format=duration',
                          '-of', 'json', str(path)], capture_output=True, text=True).stdout
    return float(json.loads(out)['format']['duration'])

def _fps(path):
    out = subprocess.run(['ffprobe', '-v', 'error', '-select_streams', 'v:0',
                          '-show_entries', 'stream=r_frame_rate', '-of', 'json',
                          str(path)], capture_output=True, text=True).stdout
    n, d = json.loads(out)['streams'][0]['r_frame_rate'].split('/')
    return float(n) / float(d)

def _res(path):
    out = subprocess.run(['ffprobe', '-v', 'error', '-select_streams', 'v:0',
                          '-show_entries', 'stream=width,height', '-of', 'json',
                          str(path)], capture_output=True, text=True).stdout
    s = json.loads(out)['streams'][0]
    return int(s['width']), int(s['height'])

def _ff(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError('ffmpeg failed:\n' + r.stderr[-1500:])

def build_sequence(idle_clip, speaking_clip, out, idle_dur, crossfade):
    for f in (idle_clip, speaking_clip):
        if not os.path.exists(f):
            raise FileNotFoundError(f'Input not found: {f}')
    os.makedirs(os.path.dirname(out) or '.', exist_ok=True)
    fps  = _fps(speaking_clip)
    W, H = _res(speaking_clip)
    tmp  = tempfile.mkdtemp()

    # Idle -> stretched to exactly idle_dur, normalized to speaking fps/res.
    k = idle_dur / _dur(idle_clip)
    idle_seg = f'{tmp}/idle_seg.mp4'
    _ff(['ffmpeg', '-y', '-i', str(idle_clip), '-filter_complex',
         f'[0:v]setpts={k:.6f}*PTS,fps={fps:.6f},scale={W}:{H},setsar=1,format=yuv420p[v]',
         '-map', '[v]', '-an', '-c:v', 'libx264', '-pix_fmt', 'yuv420p', '-crf', '18', idle_seg])

    # Speaking -> normalized only (its length IS S).
    speak_seg = f'{tmp}/speak_seg.mp4'
    _ff(['ffmpeg', '-y', '-i', str(speaking_clip), '-filter_complex',
         f'[0:v]fps={fps:.6f},scale={W}:{H},setsar=1,format=yuv420p[v]',
         '-map', '[v]', '-an', '-c:v', 'libx264', '-pix_fmt', 'yuv420p', '-crf', '18', speak_seg])
    S = _dur(speak_seg)

    if crossfade and crossfade > 0:                      # soft blends at the 2 joins
        cf = float(crossfade)
        off1 = idle_dur - cf
        off2 = idle_dur + S - 2 * cf
        fc = (f'[0:v]settb=AVTB[a];[1:v]settb=AVTB[b];[2:v]settb=AVTB[c];'
              f'[a][b]xfade=transition=fade:duration={cf:.6f}:offset={off1:.6f}[ab];'
              f'[ab][c]xfade=transition=fade:duration={cf:.6f}:offset={off2:.6f}[v]')
        mode = f'crossfade {cf:.2f}s'
    else:                                                # hard cuts, exact 6+S
        fc = '[0:v][1:v][2:v]concat=n=3:v=1[v]'
        mode = 'hard cut'

    _ff(['ffmpeg', '-y', '-i', idle_seg, '-i', speak_seg, '-i', idle_seg,
         '-filter_complex', fc, '-map', '[v]', '-an',
         '-c:v', 'libx264', '-pix_fmt', 'yuv420p', '-crf', '18', '-r', f'{fps:.6f}', str(out)])

    total = _dur(out)
    print(f'  timeline ({mode}):')
    print(f'    0.00 -> {idle_dur:5.2f} s   idle')
    print(f'    {idle_dur:5.2f} -> {idle_dur+S:5.2f} s   speaking (S={S:.2f}s)')
    print(f'    {idle_dur+S:5.2f} -> {2*idle_dur+S:5.2f} s   idle')
    print(f'  done -> {out}\n         {total:.2f}s, {W}x{H} @ {fps:.2f} fps, no audio')
    return out

print(f'Stitching  ->  idle {IDLE_DURATION}s + speaking + idle {IDLE_DURATION}s')
build_sequence(IDLE_CLIP, SPEAKING_CLIP, OUTPUT_PATH, IDLE_DURATION, CROSSFADE_SEC)


Stitching  ->  idle 3.0s + speaking + idle 3.0s
  timeline (hard cut):
    0.00 ->  3.00 s   idle
     3.00 ->  8.00 s   speaking (S=5.00s)
     8.00 -> 11.00 s   idle
  done -> ./assets/person_08_sequenced.mp4
         11.00s, 850x850 @ 30.00 fps, no audio


'./assets/person_08_sequenced.mp4'